# End-to-End Multi-Modal Inference Demo
### GNN-Based BERT for Understanding Context from Music
**Course**: CSE425 

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# Add parent directory to path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.graph_builder import build_segment_graph, compute_graph_coherence
from src.fusion_model import GNNBERTFusionModel
from src.dataset import DEFAULT_GENRES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference runtime device: {device}")

## 1. Define Input Audio Track & Semantic Text Caption
Provide a natural language music description or lyrics and inspect the paired audio graph.

In [ ]:
# Sample audio graph (8 segment nodes with 140-dim feature vectors: 128 mel + 12 chroma)
np.random.seed(101)
num_segments = 8
segment_features = np.random.randn(num_segments, 140).astype(np.float32)
# Introduce chorus recurrence between segments 2 and 6
segment_features[5] = segment_features[1] * 0.85 + np.random.randn(140) * 0.15
timestamps = [(round(s * 2.5, 2), round(s * 2.5 + 5.0, 2)) for s in range(num_segments)]

# Construct segment graph
graph = build_segment_graph(
    node_features=segment_features,
    timestamps=timestamps,
    similarity_threshold=0.65,
    track_id="demo_track_01"
)

# Natural language caption
sample_caption = "An energetic rock anthem featuring driving distorted electric guitar riffs and pounding drum grooves."
print(f"Audio Track Graph: {graph.num_nodes} nodes, {graph.edge_index.shape[1]} edges. S_graph = {compute_graph_coherence(graph):.3f}")
print(f"Context Caption: \"{sample_caption}\"")

## 2. Load Trained GNN-BERT Multi-Modal Fusion Model

In [ ]:
checkpoint_path = "../checkpoints/task3_fusion.pt"
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model = GNNBERTFusionModel(
    num_classes=len(DEFAULT_GENRES),
    fusion_mechanism="cross_attention",
    predict_emotion=True
).to(device)

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Loaded trained weights from checkpoints/task3_fusion.pt")
else:
    print("Checkpoint not found; using initialized model for demonstration.")
model.eval()

## 3. Run Inference Pipeline

In [ ]:
# Tokenize text caption
tok = tokenizer(sample_caption, padding=True, truncation=True, max_length=128, return_tensors="pt")
input_ids = tok["input_ids"].to(device)
attention_mask = tok["attention_mask"].to(device)

# Prepare graph
x = graph.x.to(device)
edge_index = graph.edge_index.to(device)

# Forward pass
with torch.no_grad():
    outputs = model(x=x, edge_index=edge_index, input_ids=input_ids, attention_mask=attention_mask)
    tag_probs = torch.sigmoid(outputs["tag_logits"]).cpu().numpy()[0]
    pred_emotion = outputs["emotion_preds"].cpu().numpy()[0]
    attn_weights = outputs["attn_weights"].cpu().numpy()[0]

print("Inference execution complete!")

## 4. Multi-Modal Visual Dashboard & Result Analysis

In [ ]:
fig = plt.figure(figsize=(16, 10))

# Subplot 1: Predicted Tag Probabilities
ax1 = fig.add_subplot(2, 2, 1)
top_indices = np.argsort(-tag_probs)[:6]
top_genres = [DEFAULT_GENRES[i] for i in top_indices]
top_scores = [tag_probs[i] for i in top_indices]
bars = ax1.barh(top_genres[::-1], top_scores[::-1], color='dodgerblue', edgecolor='black')
ax1.set_xlim(0, 1.0)
ax1.set_xlabel("Predicted Probability", fontsize=11)
ax1.set_title("Top Predicted Multi-Label Tags", fontsize=12)
for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.02, bar.get_y() + bar.get_height()/2, f"{w:.3f}", va='center', fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.4)

# Subplot 2: Continuous Emotion Circumplex (Valence & Arousal)
ax2 = fig.add_subplot(2, 2, 2)
valence, arousal = pred_emotion[0], pred_emotion[1]
ax2.set_xlim(1, 9)
ax2.set_ylim(1, 9)
ax2.axvline(5.0, color='gray', linestyle='--')
ax2.axhline(5.0, color='gray', linestyle='--')
ax2.scatter([valence], [arousal], color='crimson', s=200, zorder=5, edgecolors='black', linewidth=1.5)
ax2.text(valence + 0.2, arousal + 0.2, f"Predicted: ({valence:.2f}, {arousal:.2f})", fontsize=11, fontweight='bold')
# Circumplex quadrant labels
ax2.text(7.0, 7.5, "HAPPY / EXUBERANT\n(High Valence, High Arousal)", color='darkgreen', fontsize=9, ha='center')
ax2.text(3.0, 7.5, "ANXIOUS / ANGRY\n(Low Valence, High Arousal)", color='firebrick', fontsize=9, ha='center')
ax2.text(7.0, 2.5, "CALM / SERENE\n(High Valence, Low Arousal)", color='teal', fontsize=9, ha='center')
ax2.text(3.0, 2.5, "SAD / DEPRESSED\n(Low Valence, Low Arousal)", color='indigo', fontsize=9, ha='center')
ax2.set_xlabel("Valence (1=Negative -> 9=Positive)", fontsize=11)
ax2.set_ylabel("Arousal (1=Calm -> 9=Energetic)", fontsize=11)
ax2.set_title("Predicted Emotion on DEAM 2D Circumplex", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.3)

# Subplot 3: Cross-Attention Heatmap
ax3 = fig.add_subplot(2, 1, 2)
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
num_valid = min(len(tokens), 18)
tok_slice = tokens[:num_valid]
attn_slice = attn_weights[:num_valid].reshape(1, -1)
sns.heatmap(attn_slice, annot=True, fmt=".3f", cmap="Blues", xticklabels=tok_slice, yticklabels=["Graph Query g"], cbar=True, ax=ax3)
ax3.set_title("Cross-Attention Alignment: Audio Graph Structure Query over Caption Tokens", fontsize=12)
ax3.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()